### 登录

In [1]:
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth
import pandas as pd

def sign_in():
    with open(expanduser(r"C:\Users\nay\Desktop\qr\qr\worldquant\idcode.txt")) as f:
        credentials = json.load(f)

    username,password = credentials
    sess = requests.Session()
    sess.auth = HTTPBasicAuth(username, password)
    response = sess.post('https://api.worldquantbrain.com/authentication')
    print(response.status_code)
    print(response.json())
    return sess
sess = sign_in()

201
{'user': {'id': 'LH74870'}, 'token': {'expiry': 14400.0}, 'permissions': ['TUTORIAL']}


### 获取数据集id为fundamental6（company fundamental data for equity）下的所有数据字段

In [2]:
def get_datafields(
        s,
        searchScope,
        dataset_id:str = '',
        search: str = ''
):
    import pandas as pd
    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']
    if len(search) == 0:
        url_template = 'https://api.worldquantbrain.com/data-fields?' +\
            f"instrumentType={instrument_type}&region={region}&delay={delay}&universe={universe}&dataset.id={dataset_id}&limit=50"+\
            "&offset={x}"
        count = s.get(url_template.format(x=0)).json()['count']
    else:
        url_template = 'https://api.worldquantbrain.com/data-fields/search?' +\
            f"instrumentType={instrument_type}"+\
            f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50"+\
            f"&search={search}"+\
            "&offset={x}"
        count = 100

    datafields_list = []
    for x in range(0, count, 50):
        datafields = s.get(url_template.format(x=x))
        datafields_list.append(datafields.json()['results'])
    datafields_list_flat = [item for sublist in datafields_list for item in sublist]
    datafields_df = pd.DataFrame(datafields_list_flat)
    return datafields_df

In [3]:
searchscope = {'region':'USA','delay':'1','universe':'TOP3000','instrumentType':'EQUITY'}
# fundamental6 = get_datafields(s=sess,searchScope=searchscope,dataset_id='fundamental6')
opt8 = get_datafields(s=sess,searchScope=searchscope,dataset_id='option8')


In [4]:
# fundamental6 = fundamental6[fundamental6['type']=='MATRIX']
# fundamental6.head()
opt8 = opt8[opt8['type']=='MATRIX']
opt8.head()

,id,description,dataset,category,subcategory,region,delay,universe,type,coverage,userCount,alphaCount,themes
0,historical_volatility_10,Close-to-close Historical volatility over 10 days,"{'id': 'option8', 'name': 'Volatility Data'}","{'id': 'option', 'name': 'Option'}","{'id': 'option-option-volatility', 'name': 'Op...",USA,1,TOP3000,MATRIX,0.6983,767,2121,[]
1,historical_volatility_120,Close-to-close Historical volatility over 120 ...,"{'id': 'option8', 'name': 'Volatility Data'}","{'id': 'option', 'name': 'Option'}","{'id': 'option-option-volatility', 'name': 'Op...",USA,1,TOP3000,MATRIX,0.7004,1308,3077,[]
2,historical_volatility_150,Close-to-close Historical volatility over 150 ...,"{'id': 'option8', 'name': 'Volatility Data'}","{'id': 'option', 'name': 'Option'}","{'id': 'option-option-volatility', 'name': 'Op...",USA,1,TOP3000,MATRIX,0.7014,636,1816,[]
3,historical_volatility_180,Close-to-close Historical volatility over 180 ...,"{'id': 'option8', 'name': 'Volatility Data'}","{'id': 'option', 'name': 'Option'}","{'id': 'option-option-volatility', 'name': 'Op...",USA,1,TOP3000,MATRIX,0.7023,1555,3762,[]
4,historical_volatility_20,Close-to-close Historical volatility over 20 days,"{'id': 'option8', 'name': 'Volatility Data'}","{'id': 'option', 'name': 'Option'}","{'id': 'option-option-volatility', 'name': 'Op...",USA,1,TOP3000,MATRIX,0.6983,472,1273,[]


In [5]:
# datafields_list_fundamental6 = fundamental6['id'].values
# datafields_list_fundamental6

datafields_list_opt8 = opt8['id'].values
datafields_list_opt8

array(['historical_volatility_10', 'historical_volatility_120',
       'historical_volatility_150', 'historical_volatility_180',
       'historical_volatility_20', 'historical_volatility_30',
       'historical_volatility_60', 'historical_volatility_90',
       'implied_volatility_call_10', 'implied_volatility_call_1080',
       'implied_volatility_call_120', 'implied_volatility_call_150',
       'implied_volatility_call_180', 'implied_volatility_call_20',
       'implied_volatility_call_270', 'implied_volatility_call_30',
       'implied_volatility_call_360', 'implied_volatility_call_60',
       'implied_volatility_call_720', 'implied_volatility_call_90',
       'implied_volatility_mean_10', 'implied_volatility_mean_1080',
       'implied_volatility_mean_120', 'implied_volatility_mean_150',
       'implied_volatility_mean_180', 'implied_volatility_mean_20',
       'implied_volatility_mean_270', 'implied_volatility_mean_30',
       'implied_volatility_mean_360', 'implied_volatility_mea

### 将datafield替换到alpha模板中 group_rank({fundamental model data}/cap,subindustry),批量生产alpha

In [6]:
alpha_list = []

for datafield in datafields_list_opt8:
    print('正在將alpha表达式与setting封装')
    alpha_expression = f'group_rank({datafield}/cap,subindustry)'
    print(alpha_expression)
    simulation_data = {
    'type': 'REGULAR',
    'settings' :{
        'instrumentType':'EQUITY',
        'region':'USA',
        'universe': 'TOP3000',
        'delay' : 1,
        'decay' : 0,
        'neutralization' : 'SUBINDUSTRY',
        'truncation':  0.08,
        'pasteurization': 'ON',
        'unitHandling' : 'VERIFY',
        'nanHandling' : 'ON',
        'language' : 'FASTEXPR',
        'visualization': False,
        },
    'regular':alpha_expression
    }
    alpha_list.append(simulation_data)
print(f'一共封装了 {len(alpha_list)} 个alpha表达式')

正在將alpha表达式与setting封装
group_rank(historical_volatility_10/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_120/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_150/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_180/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_20/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_30/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_60/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(historical_volatility_90/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(implied_volatility_call_10/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(implied_volatility_call_1080/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(implied_volatility_call_120/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(implied_volatility_call_150/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(implied_volatility_call_180/cap,subindustry)
正在將alph

In [7]:
history_fields = [field for field in datafields_list_opt8 if 'historical_volatility' in field]
print(history_fields)

['historical_volatility_10', 'historical_volatility_120', 'historical_volatility_150', 'historical_volatility_180', 'historical_volatility_20', 'historical_volatility_30', 'historical_volatility_60', 'historical_volatility_90']


In [8]:
alpha_expressions = []
for field in history_fields:
    expr = f'group_zscore(ts_delta({field},60),industry)'
    alpha_expressions.append(expr)
print('total alpha expressions:', len(alpha_expressions))
alpha_expressions

total alpha expressions: 8


['group_zscore(ts_delta(historical_volatility_10,60),industry)',
 'group_zscore(ts_delta(historical_volatility_120,60),industry)',
 'group_zscore(ts_delta(historical_volatility_150,60),industry)',
 'group_zscore(ts_delta(historical_volatility_180,60),industry)',
 'group_zscore(ts_delta(historical_volatility_20,60),industry)',
 'group_zscore(ts_delta(historical_volatility_30,60),industry)',
 'group_zscore(ts_delta(historical_volatility_60,60),industry)',
 'group_zscore(ts_delta(historical_volatility_90,60),industry)']

In [ ]:
# # 更新
# days = [20,60,180,250,660]
# alpha_expressions = []
# for field in history_fields:
#     for day in days:
#          expr = f'group_zscore(ts_delta({field},{day}),industry)'
#          alpha_expressions.append(expr)
# print('total alpha expressions:', len(alpha_expressions))
# alpha_expressions

total alpha expressions: 40


['group_zscore(ts_delta(historical_volatility_10,20),industry)',
 'group_zscore(ts_delta(historical_volatility_10,60),industry)',
 'group_zscore(ts_delta(historical_volatility_10,180),industry)',
 'group_zscore(ts_delta(historical_volatility_10,250),industry)',
 'group_zscore(ts_delta(historical_volatility_10,660),industry)',
 'group_zscore(ts_delta(historical_volatility_120,20),industry)',
 'group_zscore(ts_delta(historical_volatility_120,60),industry)',
 'group_zscore(ts_delta(historical_volatility_120,180),industry)',
 'group_zscore(ts_delta(historical_volatility_120,250),industry)',
 'group_zscore(ts_delta(historical_volatility_120,660),industry)',
 'group_zscore(ts_delta(historical_volatility_150,20),industry)',
 'group_zscore(ts_delta(historical_volatility_150,60),industry)',
 'group_zscore(ts_delta(historical_volatility_150,180),industry)',
 'group_zscore(ts_delta(historical_volatility_150,250),industry)',
 'group_zscore(ts_delta(historical_volatility_150,660),industry)',
 'grou

In [9]:
alpha_list = []
for alpha_expression in alpha_expressions:
    print('正在將alpha表达式与setting封装')
    print(alpha_expression)
    simulation_data = {
    'type': 'REGULAR',
    'settings' :{
        'instrumentType':'EQUITY',
        'region':'USA',
        'universe': 'TOP3000',
        'delay' : 1,
        'decay' : 0,
        'neutralization' : 'SUBINDUSTRY',
        'truncation':  0.01,
        'pasteurization': 'ON',
        'unitHandling' : 'VERIFY',
        'nanHandling' : 'ON',
        'language' : 'FASTEXPR',
        'visualization': False,
        },
    'regular':alpha_expression
    }
    alpha_list.append(simulation_data)

正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_10,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_120,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_150,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_180,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_20,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_30,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_60,60),industry)
正在將alpha表达式与setting封装
group_zscore(ts_delta(historical_volatility_90,60),industry)


In [10]:
import re
history_fields = {
    'historical_volatility_10','historical_volatility_20','historical_volatility_30',
    'historical_volatility_60','historical_volatility_90','historical_volatility_120',
    'historical_volatility_150','historical_volatility_180','historical_volatility_210',
}
template = 'group_zscore(ts_delta({vo_alias},{delta_days}),industry)'

alpha_expressions = []
for field in history_fields:
    day_number = int(re.search(r'\d+', field).group())
    vol_alias = f'hv_{day_number}d'
    delta_days = day_number * 3
    alpha_expression = template.format(vo_alias=vol_alias, delta_days=delta_days)
    alpha_expressions.append(alpha_expression)
print('total alpha expressions:', len(alpha_expressions))
for expr in alpha_expressions:
    print(expr)

total alpha expressions: 9
group_zscore(ts_delta(hv_150d,450),industry)
group_zscore(ts_delta(hv_30d,90),industry)
group_zscore(ts_delta(hv_20d,60),industry)
group_zscore(ts_delta(hv_210d,630),industry)
group_zscore(ts_delta(hv_10d,30),industry)
group_zscore(ts_delta(hv_180d,540),industry)
group_zscore(ts_delta(hv_60d,180),industry)
group_zscore(ts_delta(hv_90d,270),industry)
group_zscore(ts_delta(hv_120d,360),industry)


In [12]:
alpha_list[1]

{'type': 'REGULAR',
 'settings': {'instrumentType': 'EQUITY',
  'region': 'USA',
  'universe': 'TOP3000',
  'delay': 1,
  'decay': 0,
  'neutralization': 'SUBINDUSTRY',
  'truncation': 0.01,
  'pasteurization': 'ON',
  'unitHandling': 'VERIFY',
  'nanHandling': 'ON',
  'language': 'FASTEXPR',
  'visualization': False},
 'regular': 'group_zscore(ts_delta(historical_volatility_120,60),industry)'}

In [ ]:
# len(alpha_list)

9

In [13]:
### 将α一个个发挥服务器回测，并检查是否断线，如果断线则重连
import logging
logging.basicConfig(filename='simulation.log', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

### 一个个发到服务器回测

In [14]:
from time import sleep
for alpha in alpha_list:
    sim_resp = sess.post(
        'https://api.worldquantbrain.com/simulations',
        json=alpha
    )
    try:
        sim_progress_url = sim_resp.headers.get('Location')
        while True:
            sim_progress_resp = sess.get(sim_progress_url)
            retry_after_sec = float(sim_progress_resp.headers.get('Retry-After', '0'))
            if retry_after_sec == 0:
                break
            sleep(retry_after_sec)
        alpha_id = sim_progress_resp.json()['alpha']
        print(f'Alpha ID: {alpha_id}')
    except :
        print(f'提交失败:等10秒后继续')
        sleep(2)

Alpha ID: A1d72Lbe
Alpha ID: d5wRPPPx
Alpha ID: P0EOALGJ
Alpha ID: 2rkN2YL6
Alpha ID: 1Y5pjnpR
Alpha ID: QPdGYGNG
Alpha ID: QPdGYwLg
Alpha ID: pwgNdAL6
